# Data-size ablation: how much reasoning data is actually needed?

Phase 3 used 40,000 training examples per language. That was a judgement call, not
a derivation. This notebook turns it into a measurement: finetune at
**5,000 / 10,000 / 20,000 / 40,000** and score each on the *same* held-out test
split, then plot accuracy against training-set size.

Each run is 3 epochs over its own subset, so smaller subsets are proportionally
faster. Total extra GPU time is roughly **three minutes per language** on a T4,
since the 40k point is just a re-run of what Phase 3 already did.

**Attach:** `lma-code` (the payload) and `lma-pretrained` (the Phase 2
checkpoints). No tokenized corpus is needed — the reasoning data is regenerated
here from a fixed seed.


In [ ]:
# ---- Run configuration -------------------------------------------------
LANG   = "hindi"          # run once per language; "hindi" then "assamese"
SIZES  = [5_000, 10_000, 20_000, 40_000]

# Held fixed across every size so the only variable is how much data is seen.
EPOCHS = 3
BATCH  = 32
LR     = 5e-5
SEED   = 3407

WORK   = "/kaggle/working"


In [ ]:
# ---- Code, data and dependencies ---------------------------------------
# The repo is private, so the code arrives as a Kaggle Dataset rather than a
# clone. Both mounts are found by content, because Kaggle derives the mount name
# from the dataset title and may keep the folder that was dragged in.
import os, sys, shutil, subprocess, pathlib

SRC = "/kaggle/working/repo"

def find_mount(marker, what):
    hits = sorted(pathlib.Path("/kaggle/input").glob(f"*/**/{marker}"),
                  key=lambda q: len(q.parts))
    if not hits:
        print("Attached datasets:", sorted(os.listdir("/kaggle/input")) or "(none)")
        raise SystemExit(f"Could not find {what} under /kaggle/input.")
    return hits[0]

code_root = find_mount("common/modeling/model.py", "the `lma-code` payload").parents[2]
ckpt_root = find_mount("hindi/best.pt", "the `lma-pretrained` checkpoints").parents[1]
print("code       ", code_root)
print("checkpoints", ckpt_root)

if os.path.exists(SRC):
    shutil.rmtree(SRC)
shutil.copytree(code_root, SRC)
os.chdir(SRC)
sys.path.insert(0, SRC)

# Drop cached common.* so a re-run picks up a newly uploaded dataset version
# instead of the module object from the previous run.
_stale = [m for m in sys.modules if m == "common" or m.startswith("common.")]
for _m in _stale:
    del sys.modules[_m]
if _stale:
    print(f"dropped {len(_stale)} cached common.* modules")

_ver = pathlib.Path(SRC) / "PAYLOAD_VERSION.txt"
print("payload version:", _ver.read_text().strip().replace("\n", " | ")
      if _ver.exists() else "UNSTAMPED")

import importlib.util
if importlib.util.find_spec("sentencepiece") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "sentencepiece"])
    importlib.invalidate_caches()

import torch
print("torch", torch.__version__, "| CUDA", torch.cuda.is_available())
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    cap = torch.cuda.get_device_capability()
    print(f"GPU: {p.name}  {p.total_memory/1e9:.1f} GB  sm_{cap[0]}{cap[1]}")
    print("training will use:",
          "bfloat16" if cap[0] >= 8 else "float16 (+GradScaler)")
else:
    print("WARNING: no GPU -- Settings > Accelerator > GPU")

PRETRAINED = str(ckpt_root / LANG / "best.pt")
print("pretrained checkpoint:", PRETRAINED)

In [ ]:
# ---- Locate the Phase 2 pretrained checkpoints --------------------------
# Found by content rather than by an exact path: Kaggle derives the mount name
# from the dataset title and silently disambiguates clashes.
import pathlib, os

_hits = sorted(pathlib.Path("/kaggle/input").glob(f"**/{LANG}/best.pt"),
               key=lambda q: len(q.parts))
if not _hits:
    print("Attached datasets:", sorted(os.listdir("/kaggle/input")) or "(none)")
    raise SystemExit(
        f"No {LANG}/best.pt found under /kaggle/input. Attach the "
        f"`lma-pretrained` dataset via Add Data.")
PRETRAINED = str(_hits[0])
print("pretrained checkpoint:", PRETRAINED)


In [ ]:
# ---- Regenerate the reasoning data --------------------------------------
# Deterministic from a fixed seed, so these are byte-identical to the splits
# committed in the repo. The test split is what every size is scored against.
import subprocess, sys, pathlib

subprocess.run([sys.executable, "-m", "common.reasoning.generate",
                "--lang", LANG], check=True)

base = pathlib.Path(f"{LANG}/train")
n_train = sum(1 for _ in open(base / "reasoning_train.jsonl", encoding="utf-8"))
n_test  = sum(1 for _ in open(base / "reasoning_test.jsonl",  encoding="utf-8"))
print(f"train {n_train:,} | test {n_test:,}")
assert n_train >= max(SIZES), f"only {n_train} training rows, need {max(SIZES)}"


In [ ]:
# ---- Build one data directory per subset size ---------------------------
# Taking the FIRST n rows is a uniform sample of the task mix, because the
# generator interleaves task types rather than writing them in blocks. Verified
# below rather than assumed: a skewed subset would confound size with mix.
import pathlib, shutil, json, collections

base = pathlib.Path(f"{LANG}/train")
train_rows = (base / "reasoning_train.jsonl").read_text(encoding="utf-8").splitlines()

for n in SIZES:
    d = pathlib.Path(f"{WORK}/subsets/{LANG}_{n}")
    d.mkdir(parents=True, exist_ok=True)
    (d / "reasoning_train.jsonl").write_text(
        "\n".join(train_rows[:n]) + "\n", encoding="utf-8")
    # val and test are identical for every size -- only the training data varies
    for s in ("val", "test", "test_unseen"):
        shutil.copy2(base / f"reasoning_{s}.jsonl", d / f"reasoning_{s}.jsonl")
    mix = collections.Counter(json.loads(l)["task"] for l in train_rows[:n])
    share = {k: round(v / n, 3) for k, v in sorted(mix.items())}
    print(f"  {n:6,} -> {share}")


In [ ]:
# ---- Finetune at each size ----------------------------------------------
# Every run starts from the same pretrained checkpoint and uses identical
# hyperparameters. Only the number of training examples changes. Steps scale
# with size because epochs are held at 3, which is the point: each subset is
# trained to its own convergence rather than for a fixed step budget.
import subprocess, sys, time, pathlib

for n in SIZES:
    out = f"{WORK}/ablation/{LANG}_{n}"
    if pathlib.Path(out, "best.pt").exists():
        print(f"{n:,}: already done, skipping")
        continue
    t0 = time.time()
    print(f"\n=== {LANG}, {n:,} training examples "
          f"({n * EPOCHS // BATCH:,} steps) ===")
    subprocess.run([sys.executable, "-m", "common.reasoning.finetune",
                    "--lang", LANG,
                    "--pretrained", PRETRAINED,
                    "--data-dir", f"{WORK}/subsets/{LANG}_{n}",
                    "--out-dir", out,
                    "--epochs", str(EPOCHS),
                    "--batch-size", str(BATCH),
                    "--lr", str(LR)], check=True)
    print(f"  {n:,} finished in {(time.time()-t0)/60:.1f} min")


In [ ]:
# ---- Score every size on the SAME held-out test split -------------------
import subprocess, sys, json, pathlib

results = {}
for n in SIZES:
    out_json = f"{WORK}/ablation/{LANG}_{n}/test_score.json"
    subprocess.run([sys.executable, "-m", "common.reasoning.evaluate",
                    "--lang", LANG,
                    "--pretrained", f"{WORK}/ablation/{LANG}_{n}/best.pt",
                    "--data-dir", f"{WORK}/subsets/{LANG}_{n}",
                    "--split", "test",
                    "--out", out_json], check=True)
    d = json.loads(pathlib.Path(out_json).read_text(encoding="utf-8"))["pretrained"]
    results[n] = {"exact_match": d["exact_match"],
                  "chance": d["chance_baseline"],
                  "by_task": {k: v["exact_match"] for k, v in d["by_task"].items()}}
    print(f"  {n:6,} -> test EM {d['exact_match']:.4f}")

    # Delete the checkpoints as soon as this size has been scored. Four sizes
    # x best.pt + last.pt is ~2.4 GB of throwaway weights, which makes the
    # output archive unusable to download for the ~100 KB actually wanted.
    for stale in pathlib.Path(f"{WORK}/ablation/{LANG}_{n}").glob("*.pt"):
        stale.unlink()

pathlib.Path(f"{WORK}/ablation/{LANG}_datasize.json").write_text(
    json.dumps(results, indent=2), encoding="utf-8")
print("\nwrote", f"{WORK}/ablation/{LANG}_datasize.json")


In [ ]:
# ---- Plot accuracy against training-set size ----------------------------
import json, pathlib
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

res = json.loads(pathlib.Path(f"{WORK}/ablation/{LANG}_datasize.json")
                 .read_text(encoding="utf-8"))
sizes = sorted(int(k) for k in res)
overall = [res[str(n)]["exact_match"] for n in sizes]
chance = res[str(sizes[0])]["chance"]
tasks = sorted(res[str(sizes[0])]["by_task"])

fig, ax = plt.subplots(figsize=(7.2, 4.6), dpi=150)
ax.plot(sizes, overall, marker="o", lw=2, color="#1f4e79", label="overall")
for t, c in zip(tasks, ("#c0392b", "#146e3c", "#b8860b")):
    ax.plot(sizes, [res[str(n)]["by_task"][t] for n in sizes],
            marker="s", ms=4, lw=1.1, alpha=0.8, color=c, label=t)
ax.axhline(chance, ls="--", lw=1, color="#888888", label=f"chance ({chance:.3f})")

ax.set_xscale("log")
ax.set_xticks(sizes)
ax.set_xticklabels([f"{n//1000}k" for n in sizes])
ax.set_title(f"{LANG.capitalize()} — reasoning accuracy vs. training-set size")
ax.set_xlabel("Training examples (log scale)")
ax.set_ylabel("Test exact match (free generation)")
ax.set_ylim(0, 1.02)
ax.grid(alpha=0.3)
ax.legend(loc="lower right", fontsize=8, frameon=True)
fig.tight_layout()

dest = pathlib.Path(f"{WORK}/ablation/datasize_curve_{LANG}.png")
fig.savefig(dest, bbox_inches="tight")
print("wrote", dest)

print("\n  size    overall   " + "  ".join(f"{t:>10s}" for t in tasks))
for n in sizes:
    r = res[str(n)]
    print(f"  {n:6,}   {r['exact_match']:.4f}   "
          + "  ".join(f"{r['by_task'][t]:10.4f}" for t in tasks))
plt.close(fig)


## After the run

Download `/kaggle/working/ablation/` (the JSON and the PNG per language). The
`.pt` checkpoints from the smaller sizes are throwaway and do not need keeping.

Then set `LANG = "assamese"` at the top and run again.

The point of the curve is where it flattens. If accuracy is already at its
plateau well before 40k, the dataset was larger than necessary and the honest
statement is that 40k was sufficient rather than optimal. If it is still rising
at 40k, more data would have helped and that is worth saying too.
